[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_05/notebook_5_5_segmentation_metrics.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# 5.5 Segmentation Metrics for Medical Imaging

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Calculate and interpret Dice coefficient** (F1 score for segmentation)
2. **Calculate and interpret IoU** (Intersection over Union / Jaccard Index)
3. **Understand Hausdorff distance** for boundary errors
4. **Calculate surface distances** (95th percentile)
5. **Handle multi-class segmentation** (per-class and average metrics)
6. **Understand inter-rater variability** and what "good" performance means

---

## Clinical Context: Why Segmentation Metrics Matter

**Journey Connections:**
- **Elena (Brain Tumor)**: Dice coefficient = 0.87 for tumor segmentation
- **Jamal (Lung Nodules)**: Measuring nodule boundaries for volume estimation

**Medical Imaging Segmentation Tasks:**
- **Tumor segmentation**: Brain tumors, lung nodules, liver lesions
- **Organ segmentation**: Heart, liver, kidneys for surgical planning
- **Pathology quantification**: Measuring infarct volume, lesion burden
- **Radiation therapy planning**: Defining treatment volumes

**Key Questions:**
- Is Dice = 0.85 "good" for brain tumor segmentation?
- What's the difference between Dice and IoU?
- Why does a small boundary error matter for surgical planning?
- How do we compare to human inter-rater variability?

**Clinical Reality:**
- Even expert radiologists disagree on exact boundaries (inter-rater Dice ~ 0.85-0.90)
- Different metrics emphasize different aspects (overlap vs boundary accuracy)
- Small errors in boundary can mean large volume errors
- Clinical use case determines acceptable performance

---

## Setup

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from scipy.spatial.distance import directed_hausdorff
from skimage import measure
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Set random seed
np.random.seed(42)

print("Setup complete!")

---

## Part 1: Understanding Core Segmentation Metrics

### Key Metrics Overview

**1. Dice Coefficient (DSC)**
- **Formula**: $\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}$
- **Range**: 0 to 1 (1 = perfect overlap)
- **Interpretation**: Harmonic mean of precision and recall for segmentation
- **Clinical meaning**: "How well do the predicted and true segmentations overlap?"

**2. IoU (Intersection over Union) / Jaccard Index**
- **Formula**: $\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$
- **Range**: 0 to 1 (1 = perfect overlap)
- **Relationship**: $\text{IoU} = \frac{\text{Dice}}{2 - \text{Dice}}$
- **Clinical meaning**: More stringent than Dice (penalizes errors more)

**3. Hausdorff Distance (HD)**
- **Formula**: Maximum distance from any boundary point to nearest point on other boundary
- **Range**: 0 to ∞ (0 = perfect boundary match)
- **Units**: Pixels or millimeters
- **Clinical meaning**: "What's the worst boundary error?"

**4. 95th Percentile Hausdorff Distance (HD95)**
- **Formula**: 95th percentile of all boundary distances
- **Range**: 0 to ∞ (0 = perfect)
- **Advantage**: Robust to outliers (single bad pixel doesn't dominate)
- **Clinical meaning**: "Typical boundary error, ignoring worst 5%"

### 1.1 Generate Synthetic Segmentation Data

In [ ]:
def create_circular_mask(size, center, radius):
    """
    Create a circular binary mask.

    Parameters:
    -----------
    size : tuple
        (height, width) of image
    center : tuple
        (y, x) center coordinates
    radius : int
        Radius of circle

    Returns:
    --------
    mask : ndarray
        Binary mask (1 inside circle, 0 outside)
    """
    Y, X = np.ogrid[:size[0], :size[1]]
    dist = np.sqrt((Y - center[0])**2 + (X - center[1])**2)
    mask = (dist <= radius).astype(np.uint8)
    return mask

def create_elliptical_mask(size, center, axes):
    """
    Create an elliptical binary mask (for tumor-like shapes).

    Parameters:
    -----------
    size : tuple
        (height, width) of image
    center : tuple
        (y, x) center coordinates
    axes : tuple
        (major_axis, minor_axis) semi-axes lengths
    """
    Y, X = np.ogrid[:size[0], :size[1]]
    dist = ((Y - center[0])**2 / axes[0]**2 +
            (X - center[1])**2 / axes[1]**2)
    mask = (dist <= 1).astype(np.uint8)
    return mask

# Create ground truth segmentation (brain tumor)
image_size = (256, 256)
gt_tumor = create_elliptical_mask(image_size, center=(128, 128), axes=(40, 30))

# Create predicted segmentation (slightly different)
pred_tumor = create_elliptical_mask(image_size, center=(130, 130), axes=(42, 28))

print(f"Ground truth tumor pixels: {gt_tumor.sum()}")
print(f"Predicted tumor pixels: {pred_tumor.sum()}")
print(f"Image size: {image_size}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(gt_tumor, cmap='gray')
axes[0].set_title('Ground Truth\n(Expert Annotation)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(pred_tumor, cmap='gray')
axes[1].set_title('Model Prediction', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Overlay: Green = TP, Red = FP, Blue = FN
overlay = np.zeros((*image_size, 3))
overlay[:, :, 1] = gt_tumor * pred_tumor  # TP (green)
overlay[:, :, 0] = pred_tumor * (1 - gt_tumor)  # FP (red)
overlay[:, :, 2] = gt_tumor * (1 - pred_tumor)  # FN (blue)

axes[2].imshow(overlay)
axes[2].set_title('Overlap Visualization\nGreen=TP, Red=FP, Blue=FN', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 1.2 Calculate Dice Coefficient

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Calculate Dice coefficient.

    Parameters:
    -----------
    y_true : ndarray
        Ground truth binary mask
    y_pred : ndarray
        Predicted binary mask
    smooth : float
        Smoothing factor to avoid division by zero

    Returns:
    --------
    dice : float
        Dice coefficient (0-1)
    """
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    intersection = np.sum(y_true_flat * y_pred_flat)
    union = np.sum(y_true_flat) + np.sum(y_pred_flat)

    dice = (2.0 * intersection + smooth) / (union + smooth)

    return dice

# Calculate Dice
dice = dice_coefficient(gt_tumor, pred_tumor)

# Calculate components
intersection = np.sum(gt_tumor * pred_tumor)
gt_size = np.sum(gt_tumor)
pred_size = np.sum(pred_tumor)

print("\nDICE COEFFICIENT CALCULATION")
print("=" * 60)
print(f"Ground truth pixels (|A|):    {gt_size}")
print(f"Predicted pixels (|B|):       {pred_size}")
print(f"Intersection (|A ∩ B|):       {intersection}")
print(f"\nDice = 2 × {intersection} / ({gt_size} + {pred_size})")
print(f"     = {2 * intersection} / {gt_size + pred_size}")
print(f"     = {dice:.4f}")
print(f"\nInterpretation:")
print(f"  Dice = {dice:.2f} → {dice*100:.1f}% overlap between prediction and ground truth")

### 1.3 Calculate IoU (Jaccard Index)

In [ ]:
def iou_score(y_true, y_pred, smooth=1e-6):
    """
    Calculate IoU (Intersection over Union) / Jaccard Index.
    """
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    intersection = np.sum(y_true_flat * y_pred_flat)
    union = np.sum(y_true_flat) + np.sum(y_pred_flat) - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou

# Calculate IoU
iou = iou_score(gt_tumor, pred_tumor)

# Calculate components
intersection = np.sum(gt_tumor * pred_tumor)
union = np.sum(gt_tumor) + np.sum(pred_tumor) - intersection

print("\nIoU (JACCARD INDEX) CALCULATION")
print("=" * 60)
print(f"Intersection (|A ∩ B|):       {intersection}")
print(f"Union (|A ∪ B|):              {union}")
print(f"\nIoU = {intersection} / {union}")
print(f"    = {iou:.4f}")
print(f"\nComparison with Dice:")
print(f"  Dice: {dice:.4f}")
print(f"  IoU:  {iou:.4f}")
print(f"\nNote: IoU is always ≤ Dice (IoU is more stringent)")
print(f"Relationship: IoU = Dice / (2 - Dice)")
print(f"Verify: {dice / (2 - dice):.4f} ≈ {iou:.4f} ✓")

### 1.4 Understanding Dice vs IoU

In [ ]:
# Plot relationship between Dice and IoU
dice_values = np.linspace(0, 1, 100)
iou_values = dice_values / (2 - dice_values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Dice vs IoU relationship
ax1.plot(dice_values, iou_values, 'b-', linewidth=2)
ax1.plot(dice_values, dice_values, 'r--', linewidth=1, label='y=x (reference)')
ax1.set_xlabel('Dice Coefficient', fontsize=12)
ax1.set_ylabel('IoU (Jaccard)', fontsize=12)
ax1.set_title('Relationship: IoU = Dice / (2 - Dice)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])

# Plot 2: Comparison table
comparison_data = []
for dice_val in [0.50, 0.70, 0.85, 0.90, 0.95]:
    iou_val = dice_val / (2 - dice_val)
    comparison_data.append([f'{dice_val:.2f}', f'{iou_val:.2f}'])

ax2.axis('tight')
ax2.axis('off')
table = ax2.table(cellText=comparison_data,
                  colLabels=['Dice', 'IoU'],
                  cellLoc='center',
                  loc='center',
                  colWidths=[0.3, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.5)

# Style header
for i in range(2):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

ax2.set_title('Dice vs IoU Comparison', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("  - Dice is always ≥ IoU (more optimistic)")
print("  - For Dice = 0.90 → IoU = 0.82 (big difference!)")
print("  - Both range 0-1, but IoU penalizes errors more")
print("  - Medical imaging typically reports Dice")
print("  - Computer vision often uses IoU")

---

## Part 2: Boundary-Based Metrics

### Why Boundary Metrics Matter

**Limitation of Dice/IoU:**
- Measure overall overlap, not boundary accuracy
- Can have high Dice but poor boundary localization
- Example: Tumor volume estimation requires accurate boundaries

**Boundary metrics answer:**
- "How far off is the predicted boundary from ground truth?"
- Critical for surgical planning, radiation therapy

### 2.1 Calculate Hausdorff Distance

In [ ]:
def get_boundary_pixels(mask):
    """
    Extract boundary pixels from binary mask.

    Uses morphological erosion to find boundary.
    """
    eroded = ndimage.binary_erosion(mask)
    boundary = mask.astype(int) - eroded.astype(int)
    return np.argwhere(boundary > 0)

def hausdorff_distance(y_true, y_pred):
    """
    Calculate Hausdorff distance between two binary masks.

    Returns:
    --------
    hd : float
        Maximum distance from any boundary point to nearest point on other boundary
    """
    # Get boundary points
    boundary_true = get_boundary_pixels(y_true)
    boundary_pred = get_boundary_pixels(y_pred)

    if len(boundary_true) == 0 or len(boundary_pred) == 0:
        return np.inf

    # Calculate directed Hausdorff distances
    hd1 = directed_hausdorff(boundary_true, boundary_pred)[0]
    hd2 = directed_hausdorff(boundary_pred, boundary_true)[0]

    # Hausdorff distance is the maximum
    hd = max(hd1, hd2)

    return hd

def hausdorff_distance_95(y_true, y_pred):
    """
    Calculate 95th percentile Hausdorff distance (more robust).
    """
    boundary_true = get_boundary_pixels(y_true)
    boundary_pred = get_boundary_pixels(y_pred)

    if len(boundary_true) == 0 or len(boundary_pred) == 0:
        return np.inf

    # Calculate all pairwise distances
    from scipy.spatial.distance import cdist

    # Distances from true boundary to predicted boundary
    dists_true_to_pred = cdist(boundary_true, boundary_pred)
    min_dists_true = np.min(dists_true_to_pred, axis=1)

    # Distances from predicted boundary to true boundary
    dists_pred_to_true = cdist(boundary_pred, boundary_true)
    min_dists_pred = np.min(dists_pred_to_true, axis=1)

    # Combine all distances
    all_dists = np.concatenate([min_dists_true, min_dists_pred])

    # 95th percentile
    hd95 = np.percentile(all_dists, 95)

    return hd95

# Calculate Hausdorff distances
hd = hausdorff_distance(gt_tumor, pred_tumor)
hd95 = hausdorff_distance_95(gt_tumor, pred_tumor)

print("\nHAUSDORFF DISTANCE")
print("=" * 60)
print(f"Hausdorff Distance (HD):      {hd:.2f} pixels")
print(f"  → Worst boundary error is {hd:.2f} pixels")
print(f"\n95th Percentile HD (HD95):    {hd95:.2f} pixels")
print(f"  → Typical boundary error is {hd95:.2f} pixels (ignoring worst 5%)")
print(f"\nInterpretation:")
print(f"  - HD is sensitive to outliers (single bad pixel)")
print(f"  - HD95 is more robust (typical performance)")
print(f"  - Lower is better (0 = perfect boundary match)")
print(f"  - Units: pixels (convert to mm using voxel spacing)")

### 2.2 Visualize Boundary Errors

In [ ]:
# Get boundaries
boundary_gt = get_boundary_pixels(gt_tumor)
boundary_pred = get_boundary_pixels(pred_tumor)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Boundary overlay
ax1 = axes[0]
ax1.imshow(np.zeros(image_size), cmap='gray', vmin=0, vmax=1)
ax1.scatter(boundary_gt[:, 1], boundary_gt[:, 0], c='green', s=1, label='Ground Truth', alpha=0.7)
ax1.scatter(boundary_pred[:, 1], boundary_pred[:, 0], c='red', s=1, label='Prediction', alpha=0.7)
ax1.set_title('Boundary Comparison\nGreen=GT, Red=Pred', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', markerscale=10)
ax1.axis('off')

# Plot 2: Distance map
ax2 = axes[1]
# Calculate distance from each pixel to GT boundary
distance_map = ndimage.distance_transform_edt(1 - gt_tumor)
# Mask to predicted region
error_map = distance_map * pred_tumor
im = ax2.imshow(error_map, cmap='hot', interpolation='nearest')
ax2.set_title('Prediction Error Map\n(Distance from GT boundary)', fontsize=14, fontweight='bold')
ax2.axis('off')
plt.colorbar(im, ax=ax2, label='Distance (pixels)')

plt.tight_layout()
plt.show()

print("\nVisualization Interpretation:")
print("  Left: Boundaries overlaid (red-green mismatch = error)")
print("  Right: Heatmap shows how far prediction is from ground truth")
print("        Brighter = larger boundary error")

---

## Part 3: Multi-Class Segmentation

### Clinical Reality: Multiple Structures

**Examples:**
- Brain tumor segmentation: Edema, enhancing tumor, necrotic core
- Cardiac MRI: Left ventricle, right ventricle, myocardium
- Organ segmentation: Liver, spleen, kidneys, pancreas

**Metrics for multi-class:**
- **Per-class Dice**: Separate Dice for each structure
- **Mean Dice**: Average across all classes
- **Weighted mean**: Weight by class importance

### 3.1 Generate Multi-Class Segmentation

In [ ]:
# Create multi-class segmentation (brain tumor with 3 classes)
# Class 0: Background
# Class 1: Edema (outer ring)
# Class 2: Tumor core

gt_multiclass = np.zeros(image_size, dtype=np.uint8)
pred_multiclass = np.zeros(image_size, dtype=np.uint8)

# Ground truth
edema_gt = create_elliptical_mask(image_size, center=(128, 128), axes=(50, 40))
core_gt = create_elliptical_mask(image_size, center=(128, 128), axes=(25, 20))
gt_multiclass[edema_gt > 0] = 1  # Edema
gt_multiclass[core_gt > 0] = 2   # Core (overwrites edema)

# Prediction (slightly off)
edema_pred = create_elliptical_mask(image_size, center=(130, 130), axes=(52, 38))
core_pred = create_elliptical_mask(image_size, center=(130, 130), axes=(27, 18))
pred_multiclass[edema_pred > 0] = 1
pred_multiclass[core_pred > 0] = 2

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Custom colormap
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['black', 'orange', 'red'])

axes[0].imshow(gt_multiclass, cmap=cmap, vmin=0, vmax=2)
axes[0].set_title('Ground Truth\nBlack=BG, Orange=Edema, Red=Core', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(pred_multiclass, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title('Prediction', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Difference map
diff = (gt_multiclass != pred_multiclass).astype(int)
axes[2].imshow(diff, cmap='gray')
axes[2].set_title('Error Map\nWhite=Mismatch', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"\nClass distribution:")
for class_id in range(3):
    gt_count = np.sum(gt_multiclass == class_id)
    pred_count = np.sum(pred_multiclass == class_id)
    class_names = ['Background', 'Edema', 'Tumor Core']
    print(f"  {class_names[class_id]:15s}: GT={gt_count:6d}, Pred={pred_count:6d}")

### 3.2 Calculate Per-Class Metrics

In [ ]:
def multiclass_dice(y_true, y_pred, num_classes):
    """
    Calculate per-class Dice coefficients.

    Parameters:
    -----------
    y_true : ndarray
        Ground truth segmentation (integer labels)
    y_pred : ndarray
        Predicted segmentation (integer labels)
    num_classes : int
        Number of classes (including background)

    Returns:
    --------
    dice_scores : dict
        Per-class Dice scores
    """
    dice_scores = {}

    for class_id in range(num_classes):
        # Binary masks for this class
        gt_binary = (y_true == class_id).astype(int)
        pred_binary = (y_pred == class_id).astype(int)

        # Calculate Dice
        dice = dice_coefficient(gt_binary, pred_binary)
        dice_scores[class_id] = dice

    return dice_scores

# Calculate per-class Dice
dice_per_class = multiclass_dice(gt_multiclass, pred_multiclass, num_classes=3)

# Calculate mean Dice (excluding background)
dice_mean = np.mean([dice_per_class[1], dice_per_class[2]])

class_names = ['Background', 'Edema', 'Tumor Core']

print("\nMULTI-CLASS SEGMENTATION METRICS")
print("=" * 60)
for class_id, name in enumerate(class_names):
    print(f"{name:15s}: Dice = {dice_per_class[class_id]:.4f}")

print(f"\nMean Dice (foreground classes): {dice_mean:.4f}")
print(f"\nInterpretation:")
print(f"  - Background Dice is usually high (lots of true negatives)")
print(f"  - Foreground classes are more challenging")
print(f"  - Report mean Dice excluding background for fair comparison")
print(f"\nElena's Brain Tumor (Journey 4):")
print(f"  AI achieved Dice = {dice_per_class[2]:.2f} for tumor core")
print(f"  Expert inter-rater Dice ~ 0.85-0.90")
print(f"  Model performance is approaching human-level!")

### 3.3 Visualize Per-Class Performance

In [ ]:
# Create bar plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Per-class Dice scores
classes = list(dice_per_class.keys())
scores = list(dice_per_class.values())
colors_bar = ['gray', 'orange', 'red']

bars = ax1.bar(class_names, scores, color=colors_bar, edgecolor='black', alpha=0.7)
ax1.set_ylabel('Dice Coefficient', fontsize=12)
ax1.set_title('Per-Class Dice Scores', fontsize=14, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{score:.3f}',
             ha='center', va='bottom', fontweight='bold')

# Plot 2: Confusion matrix-like view (per-class overlap)
data_table = []
for class_id, name in enumerate(class_names):
    gt_count = np.sum(gt_multiclass == class_id)
    pred_count = np.sum(pred_multiclass == class_id)
    overlap = np.sum((gt_multiclass == class_id) & (pred_multiclass == class_id))
    dice_val = dice_per_class[class_id]

    data_table.append([name, f'{gt_count}', f'{pred_count}', f'{overlap}', f'{dice_val:.3f}'])

ax2.axis('tight')
ax2.axis('off')
table = ax2.table(cellText=data_table,
                  colLabels=['Class', 'GT Pixels', 'Pred Pixels', 'Overlap', 'Dice'],
                  cellLoc='center',
                  loc='center',
                  colWidths=[0.2, 0.15, 0.15, 0.15, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2)

# Style header
for i in range(5):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color-code rows
for i in range(1, 4):
    table[(i, 0)].set_facecolor(colors_bar[i-1])
    table[(i, 0)].set_text_props(weight='bold', color='white' if i > 1 else 'black')

ax2.set_title('Detailed Per-Class Statistics', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

---

## Part 4: What is "Good" Performance?

### Inter-Rater Variability

**Key Insight**: Even expert radiologists disagree on exact boundaries!

**Typical inter-rater Dice scores:**
- Brain tumors: 0.85-0.92
- Cardiac chambers: 0.90-0.95
- Liver segmentation: 0.95-0.98
- Lung nodules: 0.75-0.85 (more variable)

**Implication**: Model with Dice = 0.87 might be as good as human experts!

### 4.1 Performance Benchmarks

In [ ]:
# Define performance benchmarks
benchmarks = {
    'Task': ['Brain Tumor Core', 'Brain Edema', 'Liver', 'Cardiac LV', 'Lung Nodules'],
    'Expert Inter-rater': [0.88, 0.86, 0.96, 0.92, 0.80],
    'Good Model': [0.85, 0.82, 0.94, 0.90, 0.75],
    'Acceptable Model': [0.75, 0.70, 0.88, 0.85, 0.65],
    'Clinical Use': ['Yes', 'Yes', 'Yes', 'Yes', 'Maybe']
}

df_benchmarks = pd.DataFrame(benchmarks)

print("\nSEGMENTATION PERFORMANCE BENCHMARKS (Dice Coefficient)")
print("=" * 80)
print(df_benchmarks.to_string(index=False))
print("\nInterpretation Guide:")
print("  Dice > 0.90: Excellent (approaching perfect)")
print("  Dice 0.80-0.90: Good (comparable to expert variability)")
print("  Dice 0.70-0.80: Acceptable (may be clinically useful)")
print("  Dice < 0.70: Poor (needs improvement)")
print("\nKey Point:")
print("  A model doesn't need to be PERFECT - it needs to match expert performance!")
print("  If experts have Dice = 0.85, your model with Dice = 0.83 is clinically useful.")

### 4.2 Visualize Performance Comparison

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

tasks = df_benchmarks['Task']
x = np.arange(len(tasks))
width = 0.25

bars1 = ax.bar(x - width, df_benchmarks['Expert Inter-rater'], width,
               label='Expert Inter-rater', color='green', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x, df_benchmarks['Good Model'], width,
               label='Good Model', color='blue', alpha=0.7, edgecolor='black')
bars3 = ax.bar(x + width, df_benchmarks['Acceptable Model'], width,
               label='Acceptable Model', color='orange', alpha=0.7, edgecolor='black')

ax.set_ylabel('Dice Coefficient', fontsize=12)
ax.set_title('Segmentation Performance Benchmarks by Task', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(tasks, rotation=15, ha='right')
ax.set_ylim([0, 1])
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3, axis='y')

# Add horizontal lines for reference
ax.axhline(y=0.90, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Excellent (>0.90)')
ax.axhline(y=0.70, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='Acceptable (>0.70)')

plt.tight_layout()
plt.show()

print("\nClinical Decision Framework:")
print("\n1. Compare to expert inter-rater variability")
print("   → If model Dice ≈ expert Dice, model is human-level")
print("\n2. Consider clinical use case")
print("   → Surgical planning: Need high accuracy (Dice > 0.85)")
print("   → Screening/triage: Lower threshold acceptable (Dice > 0.70)")
print("\n3. Validate on external data")
print("   → Test on different scanners, hospitals, populations")
print("\n4. Get clinical validation")
print("   → Radiologist review of predictions")
print("   → Prospective clinical trial")

---

## Key Takeaways

### 1. Core Metrics

**Dice Coefficient**
- Range: 0-1 (1 = perfect)
- Harmonic mean of precision and recall
- Most commonly reported in medical imaging
- Sensitive to both false positives and false negatives

**IoU (Jaccard Index)**
- Range: 0-1 (1 = perfect)
- More stringent than Dice (penalizes errors more)
- Relationship: IoU = Dice / (2 - Dice)
- Common in computer vision

**Hausdorff Distance**
- Units: pixels or millimeters
- Maximum boundary error
- Sensitive to outliers (one bad pixel → high HD)
- HD95 more robust (95th percentile)

### 2. Choosing Metrics

| Clinical Task | Primary Metric | Why |
|---------------|----------------|-----|
| Tumor volume estimation | Dice, IoU | Overall overlap matters |
| Surgical planning | HD95 | Boundary accuracy critical |
| Radiation therapy | Dice + HD95 | Both overlap and boundary |
| Screening/triage | Dice | Overall performance |

### 3. Multi-Class Segmentation

- Calculate per-class Dice separately
- Report mean Dice (excluding background)
- Some classes are harder than others (edema vs core)
- Weight classes by clinical importance if needed

### 4. What is "Good"?

**Compare to expert inter-rater variability:**
- Brain tumors: Dice 0.85-0.92
- Liver: Dice 0.95-0.98
- Lung nodules: Dice 0.75-0.85

**Rule of thumb:**
- Model within 0.05 of expert inter-rater → Human-level performance
- Model > 0.80 for most organs → Clinically useful
- Always validate on external data!

### 5. Common Pitfalls

- ❌ Reporting only Dice without boundary metrics
- ❌ Not comparing to inter-rater variability
- ❌ Including background class in mean Dice
- ❌ Using Hausdorff Distance without HD95 (sensitive to outliers)
- ❌ Not validating on external datasets

---

## Exercises

### Exercise 1: Metric Calculation
Create two binary masks:
- Ground truth: Circle at (64, 64) with radius 20
- Prediction: Circle at (66, 66) with radius 22

Calculate:
1. Dice coefficient
2. IoU
3. Hausdorff distance
4. How do metrics change if you shift prediction by 5 more pixels?

### Exercise 2: Understanding Dice vs IoU
1. Plot Dice vs IoU for values [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
2. Calculate the difference between Dice and IoU
3. At what Dice value is the difference largest?
4. Why does medical imaging prefer Dice?

### Exercise 3: Multi-Class Segmentation
Create a 3-class segmentation:
1. Simulate brain tumor with edema, enhancing core, necrotic core
2. Create slightly misaligned prediction
3. Calculate per-class Dice scores
4. Which class is hardest to segment? Why?

### Exercise 4: Boundary Errors
1. Create two circles that overlap perfectly
2. Gradually shift one circle away
3. Plot Dice and HD as function of shift distance
4. At what shift does Dice drop below 0.70?

### Exercise 5: Clinical Interpretation
You develop a liver segmentation model:
- Your model: Dice = 0.92, HD95 = 3.5mm
- Expert inter-rater: Dice = 0.94, HD95 = 2.8mm

Questions:
1. Is your model clinically useful?
2. What additional validation is needed?
3. For which clinical tasks would you deploy it?
4. What performance improvement would you prioritize?

---

**Next Notebook**: 5.6 Fairness Metrics and Bias Detection